In [1]:
## Load the data
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
# Generating ground truth
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
#!wget {PREFIX}/01-agentic-rag/code/rag_helper.py
#!wget {PREFIX}/04-evaluation/code/evaluation_utils.py

In [3]:
## Module Instructions:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
## For each page, build a JSON user prompt from its filename and content, 
# then call llm_structured with the Questions model. 
# Turn each returned question into a record labeled with the page's filename. 
# The call also returns the token usage, the same as in the lessons.

## Q1: Generating questions for all 72 pages costs money and takes time, so let's start small and generate questions for just the first 3 pages:

- `01-agentic-rag/lessons/01-intro.md`
- `01-agentic-rag/lessons/02-environment.md`
- `01-agentic-rag/lessons/03-rag.md`
Each call returns the token usage, which most LLM APIs report on the response object (e.g. `response.usage.input_tokens` / `prompt_tokens`).

What's the average number of input tokens across these 3 calls?

In [5]:
document_subset = documents[0:3]

In [6]:
## Structure output with pydantic
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]


In [7]:
## Load OpenAI items:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
## shape the document as json
from evaluation_utils import llm_structured_retry
import json

ground_truth = []
usages = []

for doc in document_subset:  # first 3 pages
    user_prompt = json.dumps(doc)
    result, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )
    for q in result.questions:
        ground_truth.append({"question": q, "document": doc["filename"]})
    usages.append(usage)

In [9]:
usages

[ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=117, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1137),
 ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=127, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1413),
 ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=1280, cache_write_tokens=0), output_tokens=115, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1868)]

In [10]:
avg_input_tokens = sum(u.input_tokens for u in usages) / len(usages)
print(f"Average input tokens: {avg_input_tokens}")

Average input tokens: 1353.0


## Q1 answer: 1400

### Full Ground Truth

In [11]:
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
#!wget {PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv

# Load it with pandas into a list of records called ground_truth. 
# Each record has a question and the filename of the page that should answer it.


In [36]:
import pandas as pd
ground_truth = pd.read_csv('ground-truth.csv').to_dict(orient="records")

In [13]:
ground_truth.head(2)

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md


In [38]:
# Searching through chucks
# Create the chunks:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [39]:
# Now rebuild the search from homework 2 over these chunks. 
# Build a text index (Index) and a vector index (VectorSearch), both keyed on filename. 
# Wrap each one in a function, text_search and vector_search, that takes a query and the number of results to return (5 by default).

In [40]:
from minsearch import Index
def text_search(documents, text_fields, keyword_fields, num_results=5):
    index = Index(text_fields=text_fields, keyword_fields=keyword_fields)
    index.fit(documents)

    def search(query, num_results=num_results):
        return index.search(query, num_results=num_results)

    return search

In [41]:
text_search= text_search(chunks, text_fields=["content"], keyword_fields=["filename"])

## Q2: After running text_search for it, what's the filename of the first result?

- 01-agentic-rag/lessons/01-intro.md
- 01-agentic-rag/lessons/03-rag.md
- 01-agentic-rag/lessons/13-function-calling.md
- 01-agentic-rag/lessons/10-rag-next-steps.md

In [43]:
text_search(ground_truth[0]["question"])[0]["filename"]


'01-agentic-rag/lessons/03-rag.md'

## Q2 Answer: 
`01-agentic-rag/lessons/03-rag.md`

In [23]:
import sys
sys.path.insert(0, '/workspaces/llm-zoomcamp-rag/llm-zoomcamp-onnx')

In [24]:
from embedder import Embedder
embed = Embedder('/workspaces/llm-zoomcamp-rag/llm-zoomcamp-onnx/models/Xenova/all-MiniLM-L6-v2')

2026-07-12 16:09:16.531061505 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [25]:
import numpy as np
from minsearch import VectorSearch

def vector_search(documents, embedder, keyword_fields, num_results=5):
    vectors = np.array([embedder.encode(doc['content']) for doc in documents])
    index = VectorSearch(keyword_fields=keyword_fields)
    index.fit(vectors, documents)

    def search(query, num_results=num_results):
        query_vector = embedder.encode(query)
        return index.search(query_vector, num_results=num_results)

    return search

In [27]:
vector_search = vector_search(chunks, embed, keyword_fields=["filename"])

In [28]:
#For hybrid search, reuse the rrf function from homework 2:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [29]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

## Q3: After running `vector_search` for the same question, what's the filename of the first result?

- `01-agentic-rag/lessons/01-intro.md`
- `01-agentic-rag/lessons/03-rag.md`
- `04-evaluation/lessons/11-evaluation-intro.md`
- `04-evaluation/lessons/12-rag-answers.md`

In [44]:
# Q3. First result with vector search:
vector_search(ground_truth[0]["question"])[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

## Q3 Answer: `01-agentic-rag/lessons/01-intro.md`

- compute_relevance runs search for a question and returns a list of 0s and 1s
- hit_rate is the fraction of questions where the correct page appears in the results
- mrr (Mean Reciprocal Rank) also rewards finding the page near the top
- evaluate runs a search function over the whole ground truth and returns both metrics


In [45]:
# total relevance of all ground truths also become generic
from tqdm.auto import tqdm

def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [46]:
# let's put hit rate to a function:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [47]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [48]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

## Q4: Evaluate text_search on the ground truth data.

What's the Hit Rate?
- 0.55
- 0.66
- 0.76
- 0.88

In [49]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

## Q4 Answer: hit rate ~ 0.76

## Q5: Now evaluate vector_search - the part we left for the homework, since the module only evaluated keyword search.

What's the MRR?
- 0.35
- 0.45
- 0.55
- 0.65

In [50]:
evaluate(
    ground_truth,
    vector_search
)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

## Q5 Answer: MRR ~ 0.55

## Q6: Evaluate `hybrid_search` over the full ground truth dataset for k values 1, 50, 100, and 200. Compare the MRR values for these runs.
Which k gives the best MRR?
- 1
- 50
- 100
- 200

In [51]:
# iterate through several boosts:
for k in [1, 50, 100, 200]:
    result = evaluate(
        ground_truth,
        lambda query, k=k: hybrid_search(query, k)
    )
    print(f"k={k}: {result}")

  0%|          | 0/360 [00:00<?, ?it/s]

k=1: {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}


  0%|          | 0/360 [00:00<?, ?it/s]

k=50: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

k=100: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


  0%|          | 0/360 [00:00<?, ?it/s]

k=200: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


## Q6 Answer: the best MRR was achieved with k = 1